## Calculating Crisis Density 

In [1]:
# Crisis lexicon with categories
CRISIS_LEXICON = {
    'conflict': [
        'war', 'violence', 'fight', 'attack', 'kill', 'death', 'murder', 'battle',
        'conflict', 'military', 'weapon', 'bomb', 'explosion', 'gun', 'shooting',
        'soldier', 'troop', 'casualty', 'casualties', 'fatality', 'fatalities',
        'assault', 'terrorism', 'terrorist', 'insurgent', 'rebel', 'combat',
        'warfare', 'bloodshed', 'massacre', 'genocide', 'ethnic cleansing',
        'civil war', 'armed conflict', 'hostilities', 'clashes', 'fighting',
        'casualty', 'wounded', 'injured', 'dead', 'died', 'killed', 'murdered'
    ],
    'disaster': [
        'flood', 'earthquake', 'hurricane', 'tornado', 'tsunami', 'volcano',
        'disaster', 'catastrophe', 'emergency', 'devastation', 'destruction',
        'natural disaster', 'storm', 'wildfire', 'drought', 'famine',
        'landslide', 'avalanche', 'eruption', 'tragedy', 'catastrophic',
        'relief', 'rescue', 'survivor', 'victims', 'emergency response',
        'damage', 'destroyed', 'ruined', 'collapsed', 'wreckage'
    ],
    'poverty': [
        'poverty', 'poor', 'hunger', 'starvation', 'famine', 'malnutrition',
        'destitute', 'destitution', 'homeless', 'homelessness', 'slum',
        'inequality', 'inequality', 'deprivation', 'underprivileged',
        'low income', 'poverty line', 'food insecurity', 'starving',
        'impoverished', 'needy', 'disadvantaged', 'marginalized'
    ],
    'health': [
        'disease', 'epidemic', 'pandemic', 'virus', 'infection', 'outbreak',
        'health crisis', 'medical emergency', 'plague', 'contagious',
        'quarantine', 'isolation', 'hospital', 'patient', 'treatment',
        'cure', 'vaccine', 'medication', 'symptoms', 'diagnosis',
        'public health', 'health emergency', 'sick', 'illness', 'death toll'
    ],
    'displacement': [
        'refugee', 'refugees', 'displaced', 'displacement', 'migrant', 'migration',
        'asylum', 'asylum seeker', 'flee', 'fled', 'escape', 'escaping',
        'camp', 'refugee camp', 'internally displaced', 'forced displacement',
        'exodus', 'homeland', 'homeless', 'driven out', 'expelled'
    ],
    'instability': [
        'unrest', 'protest', 'crisis', 'instability', 'political crisis',
        'turmoil', 'chaos', 'riots', 'riot', 'uprising', 'revolution',
        'coup', 'unstable', 'turmoil', 'disorder', 'anarchy', 'civil unrest',
        'political instability', 'government collapse', 'regime change',
        'demonstration', 'clashes', 'violence', 'security crisis'
    ]
}

### Crisis scoring fuction

In [2]:
import pandas as pd
import re
from collections import Counter
from tqdm import tqdm
import os

def calculate_crisis_scores(text, lexicon):
    """
    Calculate crisis density scores for a text.
    
    Args:
        text: Article text
        lexicon: Dictionary of crisis word categories
    
    Returns:
        Dictionary with crisis scores for each category
    """
    # Tokenize text (lowercase, remove punctuation)
    words = re.findall(r'\b[a-z]+\b', text.lower())
    total_words = len(words)
    
    if total_words == 0:
        return {
            'crisis_density': 0.0,
            'crisis_conflict': 0.0,
            'crisis_disaster': 0.0,
            'crisis_poverty': 0.0,
            'crisis_health': 0.0,
            'crisis_displacement': 0.0,
            'crisis_instability': 0.0
        }
    
    # Count crisis words by category
    category_counts = {}
    total_crisis_words = 0
    
    for category, word_list in lexicon.items():
        word_set = set(word_list)
        count = sum(1 for word in words if word in word_set)
        category_counts[f'crisis_{category}'] = count
        total_crisis_words += count
    
    # Calculate densities
    results = {
        'crisis_density': total_crisis_words / total_words if total_words > 0 else 0.0
    }
    
    for category, count in category_counts.items():
        results[category] = count / total_words if total_words > 0 else 0.0
    
    return results

### Processing Articles 

In [3]:
def process_crisis_scores(df, lexicon, batch_size=1000, output_dir='crisis_output'):
    """
    Process articles in batches with checkpoint saving.
    
    Args:
        df: DataFrame with articles and ids
        lexicon: Crisis word lexicon
        batch_size: Number of articles per batch
        output_dir: Directory to save checkpoints
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    total_articles = len(df)
    results = []
    
    for start_idx in tqdm(range(0, total_articles, batch_size), desc="Processing batches"):
        end_idx = min(start_idx + batch_size, total_articles)
        batch = df.iloc[start_idx:end_idx]
        
        batch_results = []
        for _, row in batch.iterrows():
            scores = calculate_crisis_scores(row['article'], lexicon)
            scores['id'] = row['id']
            batch_results.append(scores)
        
        # Save checkpoint
        checkpoint_df = pd.DataFrame(batch_results)
        checkpoint_file = os.path.join(output_dir, f'checkpoint_{end_idx:06d}.csv')
        checkpoint_df.to_csv(checkpoint_file, index=False)
        
        results.extend(batch_results)
    
    return pd.DataFrame(results)

In [8]:
from datasets import load_dataset
#  Load the data

ds = load_dataset("abisee/cnn_dailymail", "1.0.0")  # or use the parquet file
df = ds['train'].to_pandas()  
# Process with checkpoints
results_df = process_crisis_scores(df, CRISIS_LEXICON, batch_size=1000)

# Save final results
results_df.to_csv('crisis_scores_final.csv', index=False)

print(f"Processed {len(results_df)} articles")
print(f"Saved to crisis_scores_final.csv")

Processing batches: 100%|██████████| 288/288 [00:55<00:00,  5.22it/s]


Processed 287113 articles
Saved to crisis_scores_final.csv
